# Phase 2 — Text Profile Generation

**Goal:** Load `loads_monthly` (produced by Phase 1) and convert each row into a human-readable text profile.

**Why this matters:** The text profile is what gets embedded and stored in ChromaDB. Its quality directly determines retrieval quality.

**Embedding granularity:** One profile per customer × product family × month.

**Input:** `data/processed/loads_monthly.csv` (exported from `data_prep.ipynb`)  
**Output:** `data/processed/customer_profiles.csv` — `text_profile` column ready for embedding.

In [ ]:
import os
import pandas as pd

PROCESSED_DIR = "../data/processed/"

## 1. Load loads_monthly

In [ ]:
loads_monthly = pd.read_csv(os.path.join(PROCESSED_DIR, 'loads_monthly.csv'))
print(f"Loaded: {len(loads_monthly):,} rows — {loads_monthly.shape[1]} columns")
loads_monthly.sample(3)

In [ ]:
loads_monthly.loc[loads_monthly.group_name__c.str.contains('Vodafone')]['cus_name'].unique()

In [ ]:
# loads_monthly

In [ ]:
# loads_monthly.loc[loads_monthly.group_name__c.str.contains('Vodafone')]

In [ ]:
# loads_monthly.loc[
#     (loads_monthly.group_name__c.str.contains('Vodafone')) &
#     (loads_monthly.ProductFamily == 'EB') &
#     (loads_monthly.month >= '2025-01') &
#     (loads_monthly.month <= '2025-12')
# ].groupby(['month'])['total_amt'].sum()

In [ ]:
loads_monthly.loc[
    # (loads_monthly.cus_name.str.contains('VODAFONE-ΠΑΝΑΦΟΝ Α.Ε.Ε.Τ.')) &
    (loads_monthly.cus_name.str.contains('ΦΑΡΜΑΤΕΝ ΑΒΕΕ ΦΑΡΜΑΚ. ΙΑΤΡΙΚΩΝ Κ ΚΑΛΛΥΝΤΙΚΩΝ ΠΡΟΙΟΝΤΩΝ')) &
    # (loads_monthly.ProductFamily == 'EB') &
    (loads_monthly.month >= '2025-01') &
    (loads_monthly.month <= '2025-12')
# ][['total_amt','order_count']].sum()
].groupby('ProductFamily')[['total_amt','order_count']].sum()

## 2. Generate Text Profiles

Each row becomes one pipe-separated text string capturing: who the customer is, what they bought, when, and how much.

In [ ]:
def build_text_profile(row):
    employees = int(row['NumberOfEmployees']) if pd.notna(row['NumberOfEmployees']) else 'N/A'
    group_name = row['group_name__c'] if pd.notna(row['group_name__c']) else 'N/A'
    product_family = row['ProductFamily'] if pd.notna(row['ProductFamily']) else 'N/A'

    return (
        f"Customer: {row['cus_name']} | "
        f"Group: {group_name} | "
        f"Segment: {row['Segment__c']} | "
        f"Employees: {employees} | "
        f"Job Type: {row['JobType']} | "
        f"Salesperson: {row['SalesPerson']} | "
        f"Product Family: {product_family} | "
        f"Month: {row['month']} | "
        f"Revenue: \u20ac{row['total_amt']:,.0f} | "
        f"Orders: {int(row['order_count'])}"
    )

loads_monthly['text_profile'] = loads_monthly.apply(build_text_profile, axis=1)
print(f"Generated {len(loads_monthly):,} text profiles")

In [ ]:
# Inspect a few profiles
for profile in loads_monthly['text_profile'].sample(3, random_state=42):
    print(profile)
    print()

## 3. Save Output

In [ ]:
# output_path = os.path.join(PROCESSED_DIR, 'customer_profiles.csv')
# loads_monthly.to_csv(output_path, index=False)
# print(f"Saved {len(loads_monthly):,} profiles → {output_path}")